# Distracted Driver Detection — Kaggle Training
**Settings → Accelerator → GPU T4 x2** (top-right panel)

Dataset is already mounted at `/kaggle/input/competitions/state-farm-distracted-driver-detection/`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi | head -12

In [ ]:
# ── Cell 2: Clone repo ────────────────────────────────────────────────────────
# Replace YOUR_USERNAME with your actual GitHub username
!git clone https://github.com/YOUR_USERNAME/distracted-driver-detection-edge.git /kaggle/working/project
%cd /kaggle/working/project
!ls

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Kaggle already has torch/torchvision — only install the extras
!pip install -q onnx onnxruntime fastapi uvicorn pydantic python-multipart

In [ ]:
# ── Cell 4: Link dataset into project structure ───────────────────────────────
import os, sys
sys.path.insert(0, '/kaggle/working/project')

KAGGLE_INPUT = '/kaggle/input/competitions/state-farm-distracted-driver-detection'

os.makedirs('data/raw', exist_ok=True)
if not os.path.exists('data/raw/imgs'):
    os.symlink(f'{KAGGLE_INPUT}/imgs', 'data/raw/imgs')

# Verify all 10 classes
train_dir = 'data/raw/imgs/train'
total = 0
for cls in sorted(os.listdir(train_dir)):
    n = len(os.listdir(os.path.join(train_dir, cls)))
    total += n
    print(f'  {cls}: {n} images')
print(f'  TOTAL: {total}')

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────────────
# Each epoch prints its own line — you will see live output here
!python scripts/train.py \
    --epochs 20 \
    --warmup 5 \
    --batch-size 64 \
    --device cuda \
    --workers 4

In [ ]:
# ── Cell 6: Export to ONNX ────────────────────────────────────────────────────
!python scripts/export_model.py \
    --weights models/weights/best_model.pt \
    --output  /kaggle/working/driver_classifier.onnx \
    --verify

In [ ]:
# ── Cell 7: Copy weights to /kaggle/working/ for download ────────────────────
import shutil, json
shutil.copy('models/weights/best_model.pt',         '/kaggle/working/best_model.pt')
shutil.copy('models/weights/training_history.json', '/kaggle/working/training_history.json')

print('Files ready for download:')
for f in sorted(os.listdir('/kaggle/working/')):
    path = f'/kaggle/working/{f}'
    if os.path.isfile(path):
        print(f'  {f}  ({os.path.getsize(path)/1e6:.1f} MB)')

In [ ]:
# ── Cell 8: Plot training curves ──────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open('/kaggle/working/training_history.json') as f:
    history = json.load(f)

epochs     = [h['epoch'] for h in history]
train_acc  = [h['train_acc'] for h in history]
val_acc    = [h['val_acc'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_acc, 'b-o', label='Train')
ax1.plot(epochs, val_acc,   'r-o', label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.grid(True)

ax2.plot(epochs, train_loss, 'b-o', label='Train')
ax2.plot(epochs, val_loss,   'r-o', label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=120)
plt.show()

best = max(history, key=lambda h: h['val_acc'])
print(f"Best epoch: {best['epoch']}  val_acc={best['val_acc']*100:.2f}%  val_loss={best['val_loss']:.4f}")